In [3]:
import pandas as pd

# 1. Указываем точную ссылку на страницу с данными
url = "https://farside.co.uk/bitcoin-etf-flow-all-data/"

print("Загрузка данных с сайта Farside Investors...")

try:
    # 2. Считываем все HTML-таблицы со страницы. 
    # Функция возвращает список DataFrame, так как на странице может быть несколько таблиц.
    tables = pd.read_html(url)
    
    # 3. Выбираем таблицу с наибольшим количеством строк (это гарантированно наша основная таблица с данными)
    df_etf = max(tables, key=len)
    
    # 4. Очистка названий столбцов: убираем лишние пробелы и заменяем их на нижнее подчеркивание
    # Это стандартная практика, чтобы избежать ошибок при обращении к столбцам (например, df.Total_Net_Flow)
    df_etf.columns = df_etf.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
    
    # 5. Приводим столбец с датой к правильному формату datetime
    # Ищем столбец, в названии которого есть 'Date' или 'Дата'
    date_col = [col for col in df_etf.columns if 'date' in col.lower() or 'дата' in col.lower()][0]
    df_etf[date_col] = pd.to_datetime(df_etf[date_col], dayfirst=True)
    
    # Переименуем его просто в 'Date' для единообразия с другими нашими файлами
    df_etf = df_etf.rename(columns={date_col: 'Date'})
    
    # 6. Сохраняем результат в CSV
    file_name = "bitcoin_etf_flows_daily.csv"
    df_etf.to_csv(file_name, index=False, encoding='utf-8')
    
    # 7. Первичный осмотр результата
    print(f"✅ Успешно загружено и сохранено: {df_etf.shape[0]} строк, {df_etf.shape[1]} столбцов.")
    print(f"Файл сохранен как: {file_name}")
    print("\nПервые 5 строк данных:")
    display(df_etf.head())
    
    print("\nИнформация о типах данных:")
    df_etf.info()

except Exception as e:
    print(f"❌ Ошибка при загрузке данных: {e}")
    print("Возможные причины:")
    print("1. Отсутствует библиотека 'lxml' или 'html5lib'. Установите её через терминал: pip install lxml")
    print("2. Сайт временно недоступен или изменил структуру таблицы.")
    print("3. Проблемы с интернет-соединением.")

Загрузка данных с сайта Farside Investors...
❌ Ошибка при загрузке данных: HTTP Error 403: Forbidden
Возможные причины:
1. Отсутствует библиотека 'lxml' или 'html5lib'. Установите её через терминал: pip install lxml
2. Сайт временно недоступен или изменил структуру таблицы.
3. Проблемы с интернет-соединением.


In [2]:
!pip install lxml


In [4]:
import pandas as pd
import requests

# 1. Указываем ссылку и заголовки обычного браузера
url = "https://farside.co.uk/bitcoin-etf-flow-all-data/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

print("Загрузка данных с сайта Farside Investors (через requests)...")

try:
    # 2. Делаем запрос с заголовками браузера
    response = requests.get(url, headers=headers)
    response.raise_for_status()  # Проверка: если ошибка, код остановится здесь
    
    # 3. Передаем полученный HTML-текст в pandas
    tables = pd.read_html(response.text)
    
    # 4. Выбираем самую большую таблицу (это наша целевая таблица с данными)
    df_etf = max(tables, key=len)
    
    # 5. Очистка названий столбцов
    df_etf.columns = df_etf.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
    
    # 6. Преобразование даты
    date_col = [col for col in df_etf.columns if 'date' in col.lower() or 'дата' in col.lower()][0]
    df_etf[date_col] = pd.to_datetime(df_etf[date_col], dayfirst=True)
    df_etf = df_etf.rename(columns={date_col: 'Date'})
    
    # 7. Сохранение в CSV
    file_name = "bitcoin_etf_flows_daily.csv"
    df_etf.to_csv(file_name, index=False, encoding='utf-8')
    
    print(f"✅ Успешно загружено и сохранено: {df_etf.shape[0]} строк, {df_etf.shape[1]} столбцов.")
    print(f"Файл сохранен как: {file_name}")
    print("\nПервые 5 строк данных:")
    display(df_etf.head())
    
    print("\nИнформация о типах данных:")
    df_etf.info()

except requests.exceptions.RequestException as e:
    print(f"❌ Ошибка сети или блокировки: {e}")
    print("Сайт активно блокирует запросы. В этом случае мы переходим к Плану Б (см. ниже).")
except Exception as e:
    print(f"❌ Ошибка при обработке данных: {e}")

Загрузка данных с сайта Farside Investors (через requests)...
❌ Ошибка при обработке данных: time data "Total" doesn't match format "%d %b %Y", at position 651. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


C:\Users\Elviye\AppData\Local\Temp\ipykernel_11084\452382692.py:18: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [5]:
import pandas as pd
import requests
import io

# 1. Указываем ссылку и заголовки обычного браузера
url = "https://farside.co.uk/bitcoin-etf-flow-all-data/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

print("Загрузка данных с сайта Farside Investors...")

try:
    # 2. Делаем запрос с заголовками браузера
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # 3. Оборачиваем текст в StringIO, чтобы избежать FutureWarning в pandas
    tables = pd.read_html(io.StringIO(response.text))
    
    # 4. Выбираем самую большую таблицу (это наша целевая таблица с данными)
    df_etf = max(tables, key=len)
    
    # 5. Очистка названий столбцов
    df_etf.columns = df_etf.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
    
    # 6. Находим столбец с датой
    date_col = [col for col in df_etf.columns if 'date' in col.lower() or 'дата' in col.lower()][0]
    
    # 7. ПРЕОБРАЗОВАНИЕ ДАТЫ С ОБРАБОТКОЙ ОШИБОК
    # errors='coerce' превратит строки типа "Total" в NaT (Not a Time), а не вызовет краш программы
    df_etf[date_col] = pd.to_datetime(df_etf[date_col], dayfirst=True, errors='coerce')
    df_etf = df_etf.rename(columns={date_col: 'Date'})
    
    # 8. УДАЛЕНИЕ "МУСОРНЫХ" СТРОК
    # Удаляем все строки, где дата не распознулась (например, итоговая строка "Total")
    df_etf = df_etf.dropna(subset=['Date'])
    
    # 9. Сохранение в CSV
    file_name = "bitcoin_etf_flows_daily.csv"
    df_etf.to_csv(file_name, index=False, encoding='utf-8')
    
    print(f"✅ Успешно загружено и сохранено: {df_etf.shape[0]} строк, {df_etf.shape[1]} столбцов.")
    print(f"Файл сохранен как: {file_name}")
    print("\nПервые 5 строк данных:")
    display(df_etf.head())
    
    print("\nИнформация о типах данных (проверьте, что Date стал datetime64):")
    df_etf.info()

except requests.exceptions.RequestException as e:
    print(f"❌ Ошибка сети или блокировки: {e}")
except Exception as e:
    print(f"❌ Ошибка при обработке данных: {e}")

Загрузка данных с сайта Farside Investors...
✅ Успешно загружено и сохранено: 650 строк, 14 столбцов.
Файл сохранен как: bitcoin_etf_flows_daily.csv

Первые 5 строк данных:


,Date,IBIT,FBTC,BITB,ARKB,BTCO,EZBC,BRRR,HODL,BTCW,MSBT,GBTC,BTC,Total
1,2024-01-11,111.7,227.0,237.9,65.3,17.4,50.1,29.4,10.6,1.0,-,(95.1),-,655.3
2,2024-01-12,386.0,195.3,17.4,39.8,28.4,0.0,20.2,0.0,0.0,-,(484.1),-,203.0
3,2024-01-15,-,-,-,-,-,-,-,-,-,-,-,-,0.0
4,2024-01-16,212.7,102.0,50.2,122.3,31.9,0.0,15.3,7.3,0.0,-,(594.4),-,(52.7)
5,2024-01-17,371.4,358.1,68.2,50.3,57.6,1.2,1.2,4.8,1.6,-,(460.6),-,453.8



Информация о типах данных (проверьте, что Date стал datetime64):
<class 'pandas.core.frame.DataFrame'>
Index: 650 entries, 1 to 650
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    650 non-null    datetime64[ns]
 1   IBIT    650 non-null    object        
 2   FBTC    650 non-null    object        
 3   BITB    650 non-null    object        
 4   ARKB    650 non-null    object        
 5   BTCO    650 non-null    object        
 6   EZBC    650 non-null    object        
 7   BRRR    650 non-null    object        
 8   HODL    650 non-null    object        
 9   BTCW    650 non-null    object        
 10  MSBT    650 non-null    object        
 11  GBTC    650 non-null    object        
 12  BTC     650 non-null    object        
 13  Total   650 non-null    object        
dtypes: datetime64[ns](1), object(13)
memory usage: 76.2+ KB


In [6]:
import pandas as pd
import numpy as np

# 1. Загружаем только что сохраненный файл для работы с ним
df_etf = pd.read_csv("bitcoin_etf_flows_daily.csv")

# 2. Список столбцов, которые должны быть числами (все, кроме Date)
cols_to_clean = [col for col in df_etf.columns if col != 'Date']

# 3. Очищаем данные:
# Заменяем формат "(95.1)" на "-95.1" с помощью регулярного выражения
df_etf = df_etf.replace(r'\((.*?)\)', r'-\1', regex=True)

# Заменяем прочерки "-" на 0.0 (логично для дней без операций)
df_etf = df_etf.replace('-', 0.0)

# 4. Преобразуем все эти столбцы в числовой формат (float)
for col in cols_to_clean:
    # errors='coerce' превратит любые оставшиеся нечисловые символы в NaN, 
    # а fillna(0.0) заменит их на ноль для безопасности расчетов
    df_etf[col] = pd.to_numeric(df_etf[col], errors='coerce').fillna(0.0)

# 5. Для удобства переименуем столбец Total, добавив префикс ETF_
df_etf = df_etf.rename(columns={'Total': 'ETF_Total_Net_Flow'})

# 6. Сохраняем очищенную версию (чтобы не делать эту очистку каждый раз)
clean_file_name = "bitcoin_etf_flows_daily_clean.csv"
df_etf.to_csv(clean_file_name, index=False, encoding='utf-8')

print(f"✅ Данные успешно очищены и сохранены как: {clean_file_name}")
print("\nПроверка типов данных после очистки:")
df_etf.info()

print("\nПервые 5 строк (обратите внимание, что (95.1) стало -95.1):")
display(df_etf.head())

✅ Данные успешно очищены и сохранены как: bitcoin_etf_flows_daily_clean.csv

Проверка типов данных после очистки:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                650 non-null    object 
 1   IBIT                650 non-null    float64
 2   FBTC                650 non-null    float64
 3   BITB                650 non-null    float64
 4   ARKB                650 non-null    float64
 5   BTCO                650 non-null    float64
 6   EZBC                650 non-null    float64
 7   BRRR                650 non-null    float64
 8   HODL                650 non-null    float64
 9   BTCW                650 non-null    float64
 10  MSBT                650 non-null    float64
 11  GBTC                650 non-null    float64
 12  BTC                 650 non-null    float64
 13  ETF_Total_Net_Flow  650 non-null    float

,Date,IBIT,FBTC,BITB,ARKB,BTCO,EZBC,BRRR,HODL,BTCW,MSBT,GBTC,BTC,ETF_Total_Net_Flow
0,2024-01-11,111.7,227.0,237.9,65.3,17.4,50.1,29.4,10.6,1.0,0.0,-95.1,0.0,655.3
1,2024-01-12,386.0,195.3,17.4,39.8,28.4,0.0,20.2,0.0,0.0,0.0,-484.1,0.0,203.0
2,2024-01-15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-01-16,212.7,102.0,50.2,122.3,31.9,0.0,15.3,7.3,0.0,0.0,-594.4,0.0,-52.7
4,2024-01-17,371.4,358.1,68.2,50.3,57.6,1.2,1.2,4.8,1.6,0.0,-460.6,0.0,453.8


In [7]:
import pandas as pd

# 1. Загружаем очищенный файл
df_etf_final = pd.read_csv("bitcoin_etf_flows_daily_clean.csv")

# 2. Явно фиксируем единицу измерения в названиях столбцов
# Добавляем суффикс '_mln_usd', чтобы при любых расчетах было очевидно, 
# что мы работаем с миллионами долларов, а не с тысячами или единицами.
columns_to_rename = {}
for col in df_etf_final.columns:
    if col != 'Date':
        columns_to_rename[col] = f"{col}_mln_usd"

df_etf_final = df_etf_final.rename(columns=columns_to_rename)

# 3. Создаем аналитический признак: Накопительный итог (Cumulative Sum)
# Для оценки реального влияния институциональных денег на рынок важнее 
# не разовый ежедневный приток, а общий накопленный объем средств с момента запуска ETF.
df_etf_final['ETF_Cumulative_Flow_mln_usd'] = df_etf_final['Total_mln_usd'].cumsum()

# 4. Сохраняем финальную версию датасета
final_file_name = "bitcoin_etf_flows_daily_final.csv"
df_etf_final.to_csv(final_file_name, index=False, encoding='utf-8')

print(f"✅ Данные приведены к единой системе координат (млн USD) и сохранены как: {final_file_name}")
print("\nПервые 5 строк финального датасета (ключевые столбцы):")
display(df_etf_final[['Date', 'Total_mln_usd', 'ETF_Cumulative_Flow_mln_usd', 'IBIT_mln_usd']].head())

print("\nИнформация о типах данных (все числовые поля теперь float64):")
df_etf_final.info()

KeyError: 'Total_mln_usd'

In [8]:
import pandas as pd

# 1. Загружаем очищенный файл
df_etf_final = pd.read_csv("bitcoin_etf_flows_daily_clean.csv")

# 2. Находим, как на самом деле называется столбец с общей суммой
# Он может называться 'Total' или 'ETF_Total_Net_Flow' (после предыдущего шага)
total_col_name = 'ETF_Total_Net_Flow' if 'ETF_Total_Net_Flow' in df_etf_final.columns else 'Total'
print(f"Найден столбец с общей суммой: '{total_col_name}'")

# 3. Явно фиксируем единицу измерения в названиях всех числовых столбцов
columns_to_rename = {}
for col in df_etf_final.columns:
    # Пропускаем Date и столбцы, у которых суффикс уже есть (защита от двойного переименования)
    if col != 'Date' and not col.endswith('_mln_usd'):
        columns_to_rename[col] = f"{col}_mln_usd"

df_etf_final = df_etf_final.rename(columns=columns_to_rename)

# 4. Формируем новое имя для столбца общей суммы после переименования
new_total_col_name = f"{total_col_name}_mln_usd"

# 5. Создаем аналитический признак: Накопительный итог (Cumulative Sum)
# Суммируем все ежедневные потоки, чтобы видеть общий объем привлеченных/выведенных средств
df_etf_final['ETF_Cumulative_Flow_mln_usd'] = df_etf_final[new_total_col_name].cumsum()

# 6. Сохраняем финальную версию датасета
final_file_name = "bitcoin_etf_flows_daily_final.csv"
df_etf_final.to_csv(final_file_name, index=False, encoding='utf-8')

print(f"✅ Данные успешно приведены к единой системе координат и сохранены как: {final_file_name}")

print("\nПервые 5 строк финального датасета (ключевые столбцы):")
display(df_etf_final[['Date', new_total_col_name, 'ETF_Cumulative_Flow_mln_usd']].head())

print("\nПолный список столбцов для вашей проверки:")
print(df_etf_final.columns.tolist())

Найден столбец с общей суммой: 'ETF_Total_Net_Flow'
✅ Данные успешно приведены к единой системе координат и сохранены как: bitcoin_etf_flows_daily_final.csv

Первые 5 строк финального датасета (ключевые столбцы):


,Date,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2024-01-11,655.3,655.3
1,2024-01-12,203.0,858.3
2,2024-01-15,0.0,858.3
3,2024-01-16,-52.7,805.6
4,2024-01-17,453.8,1259.4



Полный список столбцов для вашей проверки:
['Date', 'IBIT_mln_usd', 'FBTC_mln_usd', 'BITB_mln_usd', 'ARKB_mln_usd', 'BTCO_mln_usd', 'EZBC_mln_usd', 'BRRR_mln_usd', 'HODL_mln_usd', 'BTCW_mln_usd', 'MSBT_mln_usd', 'GBTC_mln_usd', 'BTC_mln_usd', 'ETF_Total_Net_Flow_mln_usd', 'ETF_Cumulative_Flow_mln_usd']


In [9]:
import pandas as pd

# 1. Загружаем версию датасета, сохраненную на прошлом шаге
df_etf = pd.read_csv("bitcoin_etf_flows_daily_final.csv")

# 2. Надежный поиск нужных столбцов (на случай небольших различий в названиях)
# Ищем столбец с ежедневным общим потоком (содержит 'Total', но не 'Cumulative')
total_flow_col = [col for col in df_etf.columns if 'Total' in col and 'Cumulative' not in col][0]
cumulative_flow_col = 'ETF_Cumulative_Flow_mln_usd'

print(f"Выбраны столбцы: Date, {total_flow_col}, {cumulative_flow_col}")

# 3. Оставляем только ключевые столбцы
df_etf_clean = df_etf[['Date', total_flow_col, cumulative_flow_col]].copy()

# 4. Округляем числовые значения до 1 знака после запятой
# Это убирает визуальный шум, сохраняя достаточную точность для анализа
df_etf_clean[total_flow_col] = df_etf_clean[total_flow_col].round(1)
df_etf_clean[cumulative_flow_col] = df_etf_clean[cumulative_flow_col].round(1)

# 5. Сохраняем финальный, чистый датасет ETF
final_etf_file = "bitcoin_etf_final.csv"
df_etf_clean.to_csv(final_etf_file, index=False, encoding='utf-8')

print(f"✅ Данные успешно отфильтрованы, округлены и сохранены как: {final_etf_file}")
print("\nПервые 5 строк финального датасета:")
display(df_etf_clean.head())

print("\nРазмер датасета и типы данных:")
df_etf_clean.info()

Выбраны столбцы: Date, ETF_Total_Net_Flow_mln_usd, ETF_Cumulative_Flow_mln_usd
✅ Данные успешно отфильтрованы, округлены и сохранены как: bitcoin_etf_final.csv

Первые 5 строк финального датасета:


,Date,ETF_Total_Net_Flow_mln_usd,ETF_Cumulative_Flow_mln_usd
0,2024-01-11,655.3,655.3
1,2024-01-12,203.0,858.3
2,2024-01-15,0.0,858.3
3,2024-01-16,-52.7,805.6
4,2024-01-17,453.8,1259.4



Размер датасета и типы данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Date                         650 non-null    object 
 1   ETF_Total_Net_Flow_mln_usd   650 non-null    float64
 2   ETF_Cumulative_Flow_mln_usd  650 non-null    float64
dtypes: float64(2), object(1)
memory usage: 15.4+ KB
